In [46]:
import pandas as pd
df = pd.read_csv("papers.csv", engine="c", on_bad_lines='skip')

In [47]:
df

,id,year,title,event_type,pdf_name,abstract,paper_text
0,1,1987,Self-Organization of Associative Database and ...,NaN,1-self-organization-of-associative-database-an...,Abstract Missing,767\n\nSELF-ORGANIZATION OF ASSOCIATIVE DATABA...
1,10,1987,A Mean Field Theory of Layer IV of Visual Cort...,NaN,10-a-mean-field-theory-of-layer-iv-of-visual-c...,Abstract Missing,683\n\nA MEAN FIELD THEORY OF LAYER IV OF VISU...
2,100,1988,Storing Covariance by the Associative Long-Ter...,NaN,100-storing-covariance-by-the-associative-long...,Abstract Missing,394\n\nSTORING COVARIANCE BY THE ASSOCIATIVE\n...
3,1000,1994,Bayesian Query Construction for Neural Network...,NaN,1000-bayesian-query-construction-for-neural-ne...,Abstract Missing,Bayesian Query Construction for Neural\nNetwor...
4,1001,1994,"Neural Network Ensembles, Cross Validation, an...",NaN,1001-neural-network-ensembles-cross-validation...,Abstract Missing,"Neural Network Ensembles, Cross\nValidation, a..."
...,...,...,...,...,...,...,...
7236,994,1994,Single Transistor Learning Synapses,NaN,994-single-transistor-learning-synapses.pdf,Abstract Missing,Single Transistor Learning Synapses\n\nPaul Ha...
7237,996,1994,"Bias, Variance and the Combination of Least Sq...",NaN,996-bias-variance-and-the-combination-of-least...,Abstract Missing,"Bias, Variance and the Combination of\nLeast S..."
7238,997,1994,A Real Time Clustering CMOS Neural Engine,NaN,997-a-real-time-clustering-cmos-neural-engine.pdf,Abstract Missing,A Real Time Clustering CMOS\nNeural Engine\nT....
7239,998,1994,Learning direction in global motion: two class...,NaN,998-learning-direction-in-global-motion-two-cl...,Abstract Missing,Learning direction in global motion: two\nclas...


In [48]:
import re
import nltk
nltk.download("stopwords")
nltk.download("wordnet")
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HARGUN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HARGUN\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [49]:
stop_words = set(stopwords.words("english"))
new_words = ["fig", "figure", "image", "sample", "using", "show", "result", "large", "one", "two", "three", "four", "five", "seven", "eight", "nine", "also"]
stop_words = list(stop_words.union(new_words))

In [50]:
print(stop_words)

['more', "hadn't", 'myself', 'his', 'doesn', 'by', 'figure', 'in', 'won', 'into', 'most', 'themselves', 'with', 'shouldn', 'such', 'fig', "they've", 'here', 'out', "that'll", 'we', 'me', 'been', 'one', 'after', 'for', 'they', "she'd", 'he', 'once', 'my', 'there', 'are', 'ourselves', "wasn't", 'wasn', 'doing', "i'll", "hasn't", 'mightn', 'itself', 'no', 'yours', 'herself', 'some', 'when', "didn't", 'or', "shan't", 'don', 'again', 'through', "mightn't", 're', 'do', 'i', 've', 'as', 'nor', 'not', "mustn't", "needn't", 'what', 'd', 'ma', 'before', 'she', 'her', 'needn', 'who', "it'll", 'where', 'below', "they're", "he's", 'its', "we'll", 'can', 'nine', 'of', 'shan', 'yourselves', 'under', 'o', 'but', 'it', 'which', 'am', 'over', 'yourself', 'mustn', 'own', 'being', "i'm", "won't", 'because', 'our', "haven't", 'if', 'the', 'sample', "she's", 'that', 'only', 'few', 'too', "she'll", "it's", "he'll", 'five', 'why', 'wouldn', 'image', "i'd", 'aren', 'your', 'on', "couldn't", "shouldn't", "would

In [51]:
def preprocessor(text):
    text = text.lower()
    text = re.sub("&lt;/?.*?&gt;"," &lt;&gt; ",text)
    text = re.sub("(\\d|\\W)+"," ",text) 
    text = text.split()
    text = [word for word in text if word not in stop_words]
    text = [word for word in text if len(word) >= 3]
    lmtzr = WordNetLemmatizer()
    text = [lmtzr.lemmatize(word) for word in text]
    return " ".join(text)

In [52]:
documents = df["paper_text"].apply(lambda x:preprocessor(x))
documents

0       self organization associative database applica...
1       mean field theory layer visual cortex applicat...
2       storing covariance associative long term poten...
3       bayesian query construction neural network mod...
4       neural network ensemble cross validation activ...
                              ...                        
7236    single transistor learning synapsis paul hasle...
7237    bias variance combination least square estimat...
7238    real time clustering cmos neural engine serran...
7239    learning direction global motion class psychop...
7240    correlation interpolation network real time ex...
Name: paper_text, Length: 7241, dtype: object

In [53]:
from sklearn.feature_extraction.text import CountVectorizer
countvector = CountVectorizer(max_df = 0.95, max_features = 10000, ngram_range = (1, 3))

In [54]:
wcvector = countvector.fit_transform(documents)

In [55]:
from sklearn.feature_extraction.text import TfidfTransformer
transformer = TfidfTransformer(smooth_idf = True, use_idf = True)
transformer.fit(wcvector)

TfidfTransformer()

In [56]:
def extractVector(featurenames, sorteditems, top = 10):
    sorteditems = sorteditems[:top]
    scorevals = []
    featurevals = []
    for idx, score in sorteditems:
        fname = featurenames[idx]
        scorevals.append(round(score, 3))
        featurevals.append(featurenames[idx])

        results = {}
        for idx in range(len(featurevals)):
            results[featurevals[idx]] = scorevals[idx]
        return results

In [57]:
def sortedcoo(matrix):
    tuples = zip(matrix.col, matrix.data)
    return sorted(tuples, key = lambda x: (x[1], x[0]), reverse = True)
feature_names = countvector.get_feature_names_out()

In [61]:
def getKeywords(id, doc):
    tfidfvector = transformer.transform(countvector.transform([doc[id]]))
    sorteditems = sortedcoo(tfidfvector.tocoo())
    keywords = extractVector(feature_names, sorteditems, 10)
    return keywords

In [62]:
def printResults(id, keyword, df):
    print("\n=====Title=====")
    print(df["title"][id])
    print("\n=====Abstract=====")
    print(df["abstract"][id])
    print("\n===Keywords===")
    for k in keyword:
        print(k, keyword[k])

In [63]:
id = 941
keywords = getKeywords(id, docs)
printResults(id, keywords, df)


=====Title=====
Algorithms for Non-negative Matrix Factorization

=====Abstract=====
Non-negative matrix factorization (NMF) has previously been shown to 
be a useful decomposition for multivariate data. Two different multi- 
plicative algorithms for NMF are analyzed. They differ only slightly in 
the multiplicative factor used in the update rules. One algorithm can be 
shown to minimize the conventional least squares error while the other 
minimizes the generalized Kullback-Leibler divergence. The monotonic 
convergence of both algorithms can be proven using an auxiliary func- 
tion analogous to that used for proving convergence of the Expectation- 
Maximization algorithm. The algorithms can also be interpreted as diag- 
onally rescaled gradient descent, where the rescaling factor is optimally 
chosen to ensure convergence. 

===Keywords===
update 0.274
